In [0]:
# Databricks notebook source
# DBTITLE 1,Install Dependencies
%pip install scikit-surprise
dbutils.library.restartPython()

In [0]:
import logging
import os
import pickle

import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS_DIR    = "/Volumes/movie_recsys/data/outputs"
REVIEWS_PATH   = f"{OUTPUTS_DIR}/reviews_5core.parquet"   # key: parent_asin, user_id
SVD_MODEL_PATH = f"{OUTPUTS_DIR}/svd_model.pkl"
METRICS_PATH   = f"{OUTPUTS_DIR}/svd_metrics.json"

# ── Params ────────────────────────────────────────────────────────────────────
N_FACTORS   = 100     # blueprint spec: k=100 latent factors
N_EPOCHS    = 20      # standard for this scale
LR_ALL      = 0.005
REG_ALL     = 0.02
TEST_SIZE   = 0.20    # blueprint spec: 80/20 split
RANDOM_STATE = 42
TOP_N        = 10     # Precision@10 and NDCG@10

# Set True to retrain even if checkpoint exists
FORCE_RETRAIN = False

In [0]:
log.info("Loading reviews from %s", REVIEWS_PATH)
reviews = pd.read_parquet(REVIEWS_PATH, columns=["user_id", "parent_asin", "rating"])

assert "parent_asin" in reviews.columns, "reviews is missing parent_asin"
assert "user_id"     in reviews.columns, "reviews is missing user_id"
assert "rating"      in reviews.columns, "reviews is missing rating"

log.info("Reviews loaded: %d rows", len(reviews))
log.info("Unique users : %d", reviews["user_id"].nunique())
log.info("Unique items : %d", reviews["parent_asin"].nunique())
log.info("Rating range : %.1f – %.1f", reviews["rating"].min(), reviews["rating"].max())
log.info("Rating mean  : %.2f", reviews["rating"].mean())

In [0]:
reader  = Reader(rating_scale=(1.0, 5.0))
data    = Dataset.load_from_df(reviews[["user_id", "parent_asin", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)

log.info("Trainset: %d ratings", trainset.n_ratings)
log.info("Testset : %d ratings", len(testset))

In [0]:
if not FORCE_RETRAIN and os.path.exists(SVD_MODEL_PATH):
    log.info("SVD checkpoint found — loading model.")
    with open(SVD_MODEL_PATH, "rb") as f:
        algo = pickle.load(f)
    log.info("Model loaded from %s", SVD_MODEL_PATH)
else:
    log.info("Training SVD: n_factors=%d, n_epochs=%d, lr=%.4f, reg=%.3f",
             N_FACTORS, N_EPOCHS, LR_ALL, REG_ALL)
    algo = SVD(
        n_factors=N_FACTORS,
        n_epochs=N_EPOCHS,
        lr_all=LR_ALL,
        reg_all=REG_ALL,
        random_state=RANDOM_STATE,
        verbose=True,
    )
    algo.fit(trainset)
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    with open(SVD_MODEL_PATH, "wb") as f:
        pickle.dump(algo, f)
    log.info("SVD model saved → %s", SVD_MODEL_PATH)

In [0]:
predictions = algo.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)
log.info("RMSE : %.4f", rmse)
log.info("MAE  : %.4f", mae)

In [0]:
from collections import defaultdict

def precision_and_ndcg_at_k(predictions, k=10, threshold=3.5):
    """
    Compute Precision@k and NDCG@k across all users.
    threshold: minimum rating to count as relevant.
    """
    # Group predictions by user
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((est, true_r))

    precisions, ndcgs = [], []

    for uid, user_ratings in user_preds.items():
        # Sort by estimated rating (descending)
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        # Precision@k
        n_relevant = sum(1 for _, true_r in top_k if true_r >= threshold)
        precisions.append(n_relevant / k)

        # NDCG@k
        dcg  = sum(
            (2 ** (1 if true_r >= threshold else 0) - 1) / np.log2(rank + 2)
            for rank, (_, true_r) in enumerate(top_k)
        )
        # Ideal DCG: all relevant items at the top
        n_ideal = min(k, sum(1 for _, true_r in user_ratings if true_r >= threshold))
        idcg = sum(1 / np.log2(rank + 2) for rank in range(n_ideal))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return np.mean(precisions), np.mean(ndcgs)


precision_at_10, ndcg_at_10 = precision_and_ndcg_at_k(predictions, k=TOP_N, threshold=3.5)
log.info("Precision@10 : %.4f", precision_at_10)
log.info("NDCG@10      : %.4f", ndcg_at_10)

In [0]:
import json

metrics = {
    "model":         "SVD",
    "n_factors":     N_FACTORS,
    "n_epochs":      N_EPOCHS,
    "train_size":    trainset.n_ratings,
    "test_size":     len(testset),
    "rmse":          round(rmse, 4),
    "mae":           round(mae, 4),
    "precision_at_10": round(precision_at_10, 4),
    "ndcg_at_10":      round(ndcg_at_10, 4),
    "n_users":       reviews["user_id"].nunique(),
    "n_items":       reviews["parent_asin"].nunique(),
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
log.info("Metrics saved → %s", METRICS_PATH)

In [0]:
print("=" * 65)
print("JOB 2 VALIDATION")
print("=" * 65)

results = {}
def check(name, passed, detail=""):
    results[name] = passed
    tag = "✅" if passed else "❌"
    print(f"  {tag}  {name}" + (f"  [{detail}]" if detail else ""))

print("\nT1 · Model file")
check("svd_model.pkl exists",    os.path.exists(SVD_MODEL_PATH))
check("svd_metrics.json exists", os.path.exists(METRICS_PATH))

print("\nT2 · Model internals")
check("Has user factors",  hasattr(algo, "pu") and algo.pu.shape[1] == N_FACTORS,
      f"shape={algo.pu.shape}")
check("Has item factors",  hasattr(algo, "qi") and algo.qi.shape[1] == N_FACTORS,
      f"shape={algo.qi.shape}")

print("\nT3 · Metrics (directional benchmarks)")
check("RMSE < 1.10",          rmse < 1.10,           f"{rmse:.4f}")
check("Precision@10 > 0.05",  precision_at_10 > 0.05, f"{precision_at_10:.4f}")
check("NDCG@10 > 0.05",       ndcg_at_10 > 0.05,      f"{ndcg_at_10:.4f}")

print("\nT4 · Inference smoke test")
# Pick a known user and generate top-10 predictions
sample_user = reviews["user_id"].iloc[0]
all_items   = reviews["parent_asin"].unique()
seen_items  = set(reviews[reviews["user_id"] == sample_user]["parent_asin"])
unseen      = [i for i in all_items if i not in seen_items][:1000]
preds       = [algo.predict(sample_user, iid) for iid in unseen]
preds.sort(key=lambda x: x.est, reverse=True)
top10       = preds[:10]
check("Returns 10 predictions", len(top10) == 10)
check("Scores in [1,5] range",
      all(1.0 <= p.est <= 5.0 for p in top10),
      f"min={min(p.est for p in top10):.2f} max={max(p.est for p in top10):.2f}")

print("\n  Sample top-10 predicted items for user:", sample_user[:20], "...")
for rank, p in enumerate(top10, 1):
    print(f"    {rank:2d}. item={p.iid}  score={p.est:.3f}")

print("\n" + "=" * 65)
passed = sum(results.values())
failed = len(results) - passed
print(f"RESULT: {passed} passed, {failed} failed")
if failed == 0:
    print(f"✅ JOB 2 COMPLETE")
    print(f"   RMSE={rmse:.4f} | Precision@10={precision_at_10:.4f} | NDCG@10={ndcg_at_10:.4f}")
    print("Proceed to Job 3 (Cohort Nova simulation).")
else:
    print(f"❌ {failed} check(s) failed — see above.")
print("=" * 65)